# Notebook 4: Spatio-Temporal 3D-CNN + Bidirectional Mamba Training and Attention Visualizations

## 1. Problem Statement and Architectural Motivation

Classifying Autism Spectrum Disorder (ASD) from resting-state functional Magnetic Resonance Imaging (rs-fMRI) across international multi-site cohorts (such as the Full ABIDE I Cohort of 1,112 subjects across 17 institutions) faces two fundamental limitations in traditional deep learning pipelines:
1. **Information Loss from Temporal Mean Projection**: Collapsing the dynamic 4D blood-oxygen-level-dependent (BOLD) physiological signal $(X, Y, Z, T)$ into a static 3D anatomical volume discards transient hemodynamic anomalies, functional connectivity fluctuations, and temporal brain state transitions.
2. **Computational and Sequence Bottlenecks**: Attempting to process 4D neuroimaging with sequence models either triggers severe quadratic memory bottlenecks $O(T^2)$ and out-of-memory errors (Transformers) or suffers from limited memory horizons, gradient vanishing, and artificial unidirectional causal masks (LSTMs).

### Architectural Solution: Spatio-Temporal 3D-CNN + Bidirectional Mamba (Bi-Mamba)
This self-contained Kaggle notebook implements an end-to-end multi-GPU deep sequence pipeline:
- **Volumetric 3D-CNN Spatial Encoder**: Extracts a 128-dimensional spatial embedding $z_t \in \mathbb{R}^{128}$ from each chronological 3D volume using 4 convolutional blocks (channels: 32 -> 64 -> 128 -> 256), Batch Normalization, LeakyReLU, and 3D pooling.
- **Bidirectional Mamba (Bi-Mamba) Backbone**: Models chronological sequence interactions forward and backward with linear $O(T)$ computational and memory complexity, capturing bidirectional hemodynamic correlations with a pure PyTorch selective scan fallback.
- **Gated Temporal Attention Pooling**: Computes frame-by-frame attention weights $\alpha_t \in [0, 1]$ with $-\infty$ masking over zero-padded intervals to dynamically isolate informative neurological time frames.
- **Multi-Part Kaggle Dataset Auto-Discovery**: Natively discovers and aggregates 4D NIfTI scans and manifests across the 7 Kaggle dataset partitions (`PART_SITES`) covering all 20 ABIDE acquisition sites.
- **Multi-GPU Parallelism with Native BF16**: Accelerates training across multi-GPU setups (e.g. 4x NVIDIA L4 GPUs) using `torch.nn.DataParallel` and native BFloat16 mixed precision (`torch.amp.autocast(dtype=torch.bfloat16)`).
- **Clinical Interpretability Visualizations**: Exports out-of-fold cross-validation ROC curves (`spatiotemporal_roc_curve.png`) and plots dynamic temporal attention trajectories contrasting ASD vs. Typical Control (TC) cohorts across scan duration (`temporal_attention_trajectories.png`).

## 2. Environment Setup, Device Discovery, Multi-GPU Auto-Configuration, and Mixed Precision

We verify hardware accelerators, configure multi-GPU data parallelism (`torch.nn.DataParallel`), enable native BFloat16 Automatic Mixed Precision (`torch.amp`), and configure random seeds for strict reproducibility.

In [ ]:
import os
import sys
import math
import time
import random
import shutil
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union

# Install nibabel if missing (standard Kaggle environment may require it)
try:
    import nibabel as nib
except ImportError:
    import subprocess
    print('[INFO] Installing nibabel for 4D neuroimaging NIfTI parsing...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nibabel'], check=True)
    import nibabel as nib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# Check for compiled mamba_ssm CUDA operations, falling back to pure PyTorch reference scan
try:
    from mamba_ssm.ops.selective_scan_interface import selective_scan_fn as cuda_selective_scan_fn
    HAS_CUDA_MAMBA = True
    print('[INFO] Compiled mamba_ssm CUDA kernel detected and available.')
except (ImportError, Exception):
    HAS_CUDA_MAMBA = False
    cuda_selective_scan_fn = None
    print('[INFO] Compiled mamba-ssm kernel not found. Pure PyTorch selective scan fallback will be utilized seamlessly.')

# Reproducibility seed configuration
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# Accelerator hardware detection and Multi-GPU configuration
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
USE_DATA_PARALLEL = num_gpus > 1

print(f'[INFO] PyTorch version: {torch.__version__}')
print(f'[INFO] Primary Execution Device: {device}')
print(f'[INFO] Visible GPU count: {num_gpus}')
if num_gpus > 0:
    for i in range(num_gpus):
        gpu_name = torch.cuda.get_device_name(i)
        vram_gb = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
        print(f'       GPU {i}: {gpu_name} ({vram_gb:.2f} GB VRAM)')
    if USE_DATA_PARALLEL:
        print(f'[INFO] Multi-GPU DataParallel enabled across all {num_gpus} devices.')
else:
    print('[WARNING] CUDA is not available. Execution will proceed on CPU.')

# Native BF16 mixed precision check (supported natively on NVIDIA L4, Ampere, Ada, Hopper)
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f'[INFO] Native BF16 mixed precision supported and enabled: {USE_BF16}')

## 3. Multi-Part Kaggle Dataset Auto-Discovery & Partition Registry

The Full ABIDE I raw 4D rs-fMRI dataset is partitioned into 7 modular parts (`PART_SITES`) to strictly comply with Kaggle storage boundaries (~20 GB).

### Kaggle Dataset Part Registry (`PART_SITES`)
- **Part 1**: `['NYU']` (184 subjects)
- **Part 2**: `['UM_1']` (110 subjects)
- **Part 3**: `['USM', 'OHSU', 'CALTECH']` (167 subjects)
- **Part 4**: `['UCLA_1', 'PITT', 'MAX_MUN', 'UCLA_2']` (200 subjects)
- **Part 5**: `['YALE', 'UM_2', 'CMU', 'SBL']` (148 subjects)
- **Part 6**: `['KKI', 'LEUVEN_2', 'STANFORD', 'LEUVEN_1']` (194 subjects)
- **Part 7**: `['OLIN', 'TRINITY', 'SDSU']` (121 subjects)

Each mounted dataset part contains:
```text
abide_i_raw_partX/  (or dataset root)
|-- metadata.csv
|-- dataset-metadata.json
`-- scans/
    `-- {file_id}_func_preproc.nii.gz
```
The auto-discovery function recursively inspects `/kaggle/input/` and local paths, finds all `metadata.csv` files, resolves paths to actual NIfTI files on disk, and combines them into a unified cohort manifest.

In [ ]:
# Canonical mapping of the 7 Kaggle dataset parts to the 20 ABIDE site identifiers
PART_SITES: Dict[int, List[str]] = {
    1: ['NYU'],
    2: ['UM_1'],
    3: ['USM', 'OHSU', 'CALTECH'],
    4: ['UCLA_1', 'PITT', 'MAX_MUN', 'UCLA_2'],
    5: ['YALE', 'UM_2', 'CMU', 'SBL'],
    6: ['KKI', 'LEUVEN_2', 'STANFORD', 'LEUVEN_1'],
    7: ['OLIN', 'TRINITY', 'SDSU'],
}

# Create reverse lookup from site to part number
SITE_TO_PART: Dict[str, int] = {}
for part_num, sites in PART_SITES.items():
    for site in sites:
        SITE_TO_PART[site] = part_num

def discover_multisite_dataset(
    candidate_paths: Optional[Sequence[Union[str, Path]]] = None,
    recursive: bool = True,
) -> Tuple[pd.DataFrame, Dict[str, Path]]:
    """Dynamically discover phenotypic manifest and 4D functional NIfTI files across parts.
    
    Supports both single-part and multi-part Kaggle mounted datasets, resolving relative paths
    and pairing each subject with its corresponding 4D NIfTI file on disk.
    """
    if candidate_paths is None:
        search_roots = [
            Path('/kaggle/input'),
            Path('data'),
            Path('notebooks/data'),
            Path('notebooks/abide_data'),
        ]
    else:
        search_roots = [Path(p) for p in candidate_paths]

    existing_roots: List[Path] = []
    seen_roots = set()
    for p in search_roots:
        if p.exists():
            res = p.resolve()
            if res not in seen_roots:
                seen_roots.add(res)
                existing_roots.append(p)

    if not existing_roots:
        raise FileNotFoundError(f'None of the candidate paths exist: {[str(p) for p in search_roots]}')

    discovered_csvs: List[Path] = []
    for root in existing_roots:
        if root.is_file() and root.suffix == '.csv':
            discovered_csvs.append(root)
            continue
        pattern = '**/*.csv' if recursive else '*.csv'
        for candidate in root.glob(pattern):
            lower_name = candidate.name.lower()
            if 'phenotypic' in lower_name or 'metadata' in lower_name:
                discovered_csvs.append(candidate)

    unique_csvs: List[Path] = []
    seen_csv_paths = set()
    for c in discovered_csvs:
        res = c.resolve()
        if res not in seen_csv_paths:
            seen_csv_paths.add(res)
            unique_csvs.append(c)

    if not unique_csvs:
        raise FileNotFoundError(
            'Could not locate phenotypic CSV across candidate paths: '
            f'{[str(p) for p in existing_roots]}'
        )

    # Collect functional NIfTI acquisitions across existing roots
    nifti_files_set = set()
    for root in existing_roots:
        pattern = '**/*func_preproc.nii*' if recursive else '*func_preproc.nii*'
        for nifti_path in root.glob(pattern):
            nifti_files_set.add(nifti_path.resolve())

    nifti_files: List[Path] = sorted(list(nifti_files_set))

    dfs: List[pd.DataFrame] = []
    for csv_path in unique_csvs:
        try:
            raw_df = pd.read_csv(csv_path)
        except Exception:
            continue
        if raw_df.empty:
            continue

        df_part = raw_df.copy()
        col_map = {col: col.upper() for col in df_part.columns}
        rev_col_map = {v: k for k, v in col_map.items()}

        sub_col = rev_col_map.get('SUB_ID') or rev_col_map.get('SUBJECT_ID')
        if sub_col is None:
            continue
        df_part['subject_id'] = df_part[sub_col].astype(str).str.strip()

        dx_col = rev_col_map.get('DX_GROUP') or rev_col_map.get('DX') or rev_col_map.get('LABEL')
        if dx_col is not None:
            unique_vals = set(df_part[dx_col].dropna().unique())
            if {1, 2}.issubset(unique_vals) or 2 in unique_vals:
                df_part['label'] = (df_part[dx_col] == 1).astype(int)
            else:
                df_part['label'] = df_part[dx_col].astype(int)
        else:
            continue

        site_col = rev_col_map.get('SITE_ID') or rev_col_map.get('SITE')
        df_part['site_id'] = df_part[site_col].astype(str).str.strip() if site_col else 'UNKNOWN'

        file_id_col = rev_col_map.get('FILE_ID')
        df_part['file_id'] = df_part[file_id_col].astype(str).str.strip() if file_id_col else 'no_filename'

        # Resolve file paths if provided in metadata
        fp_col = rev_col_map.get('FILE_PATH')
        if fp_col:
            resolved_paths: List[Optional[Path]] = []
            for _, r in df_part.iterrows():
                val = str(r[fp_col]).strip() if pd.notna(r[fp_col]) else ''
                candidate_target: Optional[Path] = None
                if val:
                    p_val = Path(val)
                    if p_val.is_absolute() and p_val.exists():
                        candidate_target = p_val
                    elif (csv_path.parent / p_val).exists():
                        candidate_target = (csv_path.parent / p_val).resolve()
                    elif (csv_path.parent / 'scans' / p_val.name).exists():
                        candidate_target = (csv_path.parent / 'scans' / p_val.name).resolve()
                    elif (csv_path.parent.parent / 'scans' / p_val.name).exists():
                        candidate_target = (csv_path.parent.parent / 'scans' / p_val.name).resolve()
                resolved_paths.append(candidate_target)
            df_part['file_path'] = resolved_paths
        else:
            df_part['file_path'] = None

        dfs.append(df_part)

    if not dfs:
        raise ValueError('Could not extract valid subject records from candidate CSVs.')

    df = pd.concat(dfs, ignore_index=True)
    df.drop_duplicates(subset=['subject_id'], keep='first', inplace=True)

    # Match unmapped subjects against discovered NIfTI files
    file_map: Dict[str, Path] = {}
    for _, row in df.iterrows():
        sid = row['subject_id']
        if row.get('file_path') is not None and pd.notna(row['file_path']):
            file_map[sid] = Path(row['file_path'])

    if nifti_files:
        unmatched_rows = df[df['subject_id'].map(lambda s: s not in file_map)]
        import re
        for nifti in nifti_files:
            name = nifti.name
            for _, row in unmatched_rows.iterrows():
                sub_id = row['subject_id']
                file_id = row['file_id']
                matched = False
                if file_id != 'no_filename' and file_id in name:
                    matched = True
                elif f'00{sub_id}' in name or f'_{sub_id}_' in name or name.startswith(f'{sub_id}_'):
                    matched = True
                elif re.search(rf'(?:^|[^0-9])0*{re.escape(sub_id)}(?:[^0-9]|$)', name):
                    matched = True
                if matched:
                    file_map[sub_id] = nifti.resolve()
                    break

    df['file_path'] = df['subject_id'].map(lambda sid: file_map.get(sid))
    matched_df = df[df['file_path'].notna()].copy()
    if matched_df.empty:
        raise FileNotFoundError(
            'Could not locate matching functional NIfTI files for subjects in manifest '
            f'across candidate paths: {[str(p) for p in existing_roots]}'
        )

    matched_df['file_path'] = matched_df['file_path'].map(lambda p: Path(p))
    matched_df['part_num'] = matched_df['site_id'].map(lambda s: SITE_TO_PART.get(s, -1))
    matched_df.reset_index(drop=True, inplace=True)

    return matched_df, {row['subject_id']: Path(row['file_path']) for _, row in matched_df.iterrows()}

## 4. Spatio-Temporal Data Loader and Neuroimaging Preprocessing

4D rs-fMRI neuroimaging data exhibits variable acquisition lengths ($T \in [78, 300]$) and scanner-dependent baseline differences. We apply:
1. **Scanner Buffer Trimming**: Discards first 10 and last 10 temporal frames to remove scanner magnetization stabilization noise and termination artifacts.
2. **Subject-Level Non-Zero Voxel Intensity Normalization**: Computes z-score intensity normalization strictly on brain tissue voxels ($v \neq 0$), preserving zero air background.
3. **Spatial Standardization to $64 \times 64 \times 36$**: Center crops/pads or resamples 3D spatial grids to uniform dimensions matching the 4-block 3D-CNN receptive field.
4. **Temporal Padding to $T_{\max} = 200$ with Boolean Masks**: Padded frames receive value $0$ and boolean mask `False` so that sequence modeling and attention pooling strictly ignore non-biological padding.

In [ ]:
def trim_temporal_frames(
    volume: Union[np.ndarray, torch.Tensor],
    trim_start: int = 10,
    trim_end: int = 10,
    time_axis: int = 0,
) -> Union[np.ndarray, torch.Tensor]:
    """Trim boundary temporal frames to remove scanner equilibrium and termination artifacts."""
    total_frames = volume.shape[time_axis]
    min_required = trim_start + trim_end
    if total_frames <= min_required:
        raise ValueError(
            f'Cannot trim {trim_start} start and {trim_end} end frames: '
            f'volume temporal axis {time_axis} only has {total_frames} frames (need > {min_required}).'
        )
    end_idx = total_frames - trim_end
    axis = time_axis if time_axis >= 0 else volume.ndim + time_axis
    slices = [slice(None)] * volume.ndim
    slices[axis] = slice(trim_start, end_idx)
    return volume[tuple(slices)]

def normalize_nonzero_voxels(
    volume: Union[np.ndarray, torch.Tensor],
    eps: float = 1e-8,
) -> Union[np.ndarray, torch.Tensor]:
    """Apply subject-level z-score intensity normalization strictly across non-zero voxels."""
    if isinstance(volume, torch.Tensor):
        mask = volume != 0
        if not torch.any(mask):
            return volume.clone()
        voxels = volume[mask].to(torch.float32)
        mean_val = torch.mean(voxels)
        std_val = torch.std(voxels)
        std_denom = std_val if std_val > eps else torch.tensor(1.0, dtype=mean_val.dtype, device=mean_val.device)
        normalized = torch.zeros_like(volume, dtype=torch.float32)
        normalized[mask] = (voxels - mean_val) / std_denom
        return normalized.to(volume.dtype)

    mask_np = volume != 0
    if not np.any(mask_np):
        return volume.copy()
    voxels_np = volume[mask_np].astype(np.float32)
    mean_val_np = np.mean(voxels_np)
    std_val_np = np.std(voxels_np)
    std_denom_np = std_val_np if std_val_np > eps else 1.0
    normalized_np = np.zeros_like(volume, dtype=np.float32)
    normalized_np[mask_np] = (voxels_np - mean_val_np) / std_denom_np
    return normalized_np.astype(volume.dtype)

def standardize_spatial_volume(
    volume: Union[np.ndarray, torch.Tensor],
    target_shape: Tuple[int, int, int] = (64, 64, 36),
    mode: str = 'crop_pad',
) -> Union[np.ndarray, torch.Tensor]:
    """Standardize 3D spatial dimensions (trailing 3 axes) to target resolution."""
    is_torch = isinstance(volume, torch.Tensor)
    original_ndim = volume.ndim
    if original_ndim < 3:
        raise ValueError(f'Expected at least 3 dimensions for spatial volume, got shape {volume.shape}')

    current_shape = volume.shape[-3:]
    if tuple(current_shape) == tuple(target_shape):
        return volume

    if mode == 'resample':
        tensor_vol: torch.Tensor = volume if isinstance(volume, torch.Tensor) else torch.from_numpy(volume)
        orig_dtype = tensor_vol.dtype
        leading_shape = tensor_vol.shape[:-3]
        num_vols = int(np.prod(leading_shape)) if leading_shape else 1
        reshaped = tensor_vol.reshape(num_vols, 1, *current_shape).to(torch.float32)
        resampled = F.interpolate(reshaped, size=target_shape, mode='trilinear', align_corners=False).to(orig_dtype)
        final_shape = (*leading_shape, *target_shape)
        output_tensor = resampled.reshape(final_shape)
        return output_tensor if is_torch else output_tensor.cpu().numpy()

    # Symmetric center-crop and zero-padding mode
    if isinstance(volume, torch.Tensor):
        processed_t: torch.Tensor = volume
        for axis_offset in range(3):
            axis = original_ndim - 3 + axis_offset
            curr_size = processed_t.shape[axis]
            targ_size = target_shape[axis_offset]
            if curr_size == targ_size:
                continue
            elif curr_size < targ_size:
                pad_front = (targ_size - curr_size) // 2
                pad_back = targ_size - curr_size - pad_front
                pad_tuple = [0, 0] * original_ndim
                rev_axis = original_ndim - 1 - axis
                pad_tuple[rev_axis * 2] = pad_front
                pad_tuple[rev_axis * 2 + 1] = pad_back
                processed_t = F.pad(processed_t, tuple(pad_tuple), mode='constant', value=0)
            else:
                crop_start = (curr_size - targ_size) // 2
                crop_end = crop_start + targ_size
                slices = [slice(None)] * original_ndim
                slices[axis] = slice(crop_start, crop_end)
                processed_t = processed_t[tuple(slices)]
        return processed_t

    processed_np: np.ndarray = volume
    for axis_offset in range(3):
        axis = original_ndim - 3 + axis_offset
        curr_size = processed_np.shape[axis]
        targ_size = target_shape[axis_offset]
        if curr_size == targ_size:
            continue
        elif curr_size < targ_size:
            pad_front = (targ_size - curr_size) // 2
            pad_back = targ_size - curr_size - pad_front
            pad_width = [(0, 0)] * original_ndim
            pad_width[axis] = (pad_front, pad_back)
            processed_np = np.pad(processed_np, pad_width, mode='constant', constant_values=0)
        else:
            crop_start = (curr_size - targ_size) // 2
            crop_end = crop_start + targ_size
            slices = [slice(None)] * original_ndim
            slices[axis] = slice(crop_start, crop_end)
            processed_np = processed_np[tuple(slices)]
    return processed_np

def pad_temporal_sequence(
    sequence: Union[np.ndarray, torch.Tensor],
    target_length: int = 200,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Pad or truncate variable temporal sequence to target duration with boolean mask."""
    tensor_seq = sequence if isinstance(sequence, torch.Tensor) else torch.from_numpy(sequence)
    curr_t = tensor_seq.shape[0]
    if curr_t >= target_length:
        truncated = tensor_seq[:target_length]
        mask = torch.ones(target_length, dtype=torch.bool, device=tensor_seq.device)
        return truncated, mask
    pad_shape = (target_length - curr_t, *tensor_seq.shape[1:])
    zeros = torch.zeros(pad_shape, dtype=tensor_seq.dtype, device=tensor_seq.device)
    padded = torch.cat([tensor_seq, zeros], dim=0)
    mask = torch.cat([
        torch.ones(curr_t, dtype=torch.bool, device=tensor_seq.device),
        torch.zeros(target_length - curr_t, dtype=torch.bool, device=tensor_seq.device),
    ], dim=0)
    return padded, mask

class SpatioTemporalFMRIDataset(Dataset):
    """PyTorch Dataset for 4D rs-fMRI neuroimaging sequences."""
    def __init__(
        self,
        manifest: Union[pd.DataFrame, str, Path],
        target_spatial_shape: Tuple[int, int, int] = (64, 64, 36),
        target_seq_len: int = 200,
        trim_start: int = 10,
        trim_end: int = 10,
        normalize: bool = True,
        spatial_mode: str = 'crop_pad',
    ) -> None:
        if isinstance(manifest, (str, Path)):
            self.manifest = pd.read_csv(manifest)
        elif isinstance(manifest, pd.DataFrame):
            self.manifest = manifest.copy().reset_index(drop=True)
        else:
            raise TypeError(f'Expected DataFrame or Path, got {type(manifest)}')
        self.target_spatial_shape = target_spatial_shape
        self.target_seq_len = target_seq_len
        self.trim_start = trim_start
        self.trim_end = trim_end
        self.normalize = normalize
        self.spatial_mode = spatial_mode

    def __len__(self) -> int:
        return len(self.manifest)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.manifest.iloc[idx]
        file_path = Path(row['file_path'])
        if not file_path.exists():
            raise FileNotFoundError(f'fMRI NIfTI file not found: {file_path}')

        raw_img = nib.load(str(file_path))
        canonical_img = nib.as_closest_canonical(raw_img)
        data = np.asarray(canonical_img.get_fdata(), dtype=np.float32)

        if data.ndim == 4:
            time_axis = 0 if data.shape[0] > data.shape[-1] and data.shape[-1] in [36, 61, 64] else -1
        else:
            raise ValueError(f'Expected 4D NIfTI scan, got shape {data.shape}')

        trimmed = trim_temporal_frames(data, trim_start=self.trim_start, trim_end=self.trim_end, time_axis=time_axis)
        trimmed_np = trimmed if isinstance(trimmed, np.ndarray) else trimmed.cpu().numpy()

        if time_axis == -1 or time_axis == 3:
            seq_data = np.moveaxis(trimmed_np, -1, 0)
        else:
            seq_data = trimmed_np

        if self.normalize:
            normalized = normalize_nonzero_voxels(seq_data)
            seq_data = normalized if isinstance(normalized, np.ndarray) else normalized.cpu().numpy()

        standardized = standardize_spatial_volume(seq_data, target_shape=self.target_spatial_shape, mode=self.spatial_mode)
        tensor_seq = torch.from_numpy(standardized).unsqueeze(1).to(torch.float32) if isinstance(standardized, np.ndarray) else standardized.unsqueeze(1).to(torch.float32)
        padded_vol, temporal_mask = pad_temporal_sequence(tensor_seq, target_length=self.target_seq_len)

        return {
            'volume': padded_vol,
            'mask': temporal_mask,
            'label': torch.tensor(int(row['label']), dtype=torch.long),
            'subject_id': str(row.get('subject_id', idx)),
            'site_id': str(row.get('site_id', 'UNKNOWN')),
        }

def spatiotemporal_collate_fn(batch: Sequence[Dict[str, Any]]) -> Dict[str, Any]:
    """Collate individual subject dictionaries into batched PyTorch tensors."""
    return {
        'volume': torch.stack([item['volume'] for item in batch], dim=0),
        'mask': torch.stack([item['mask'] for item in batch], dim=0),
        'label': torch.stack([item['label'] for item in batch], dim=0),
        'subject_ids': [item['subject_id'] for item in batch],
        'site_ids': [item['site_id'] for item in batch],
    }

## 5. Spatio-Temporal 3D-CNN + Bidirectional Mamba Architecture

The architecture consists of four modular, decoupled components:
1. **`Spatial3DCNNEncoder`**: Maps $(B \times T, 1, 64, 64, 36) \to (B, T, 128)$ via 4 3D convolutional blocks with Batch Normalization, LeakyReLU, and 3D Max Pooling.
2. **`selective_scan_fn` & `MambaBlock`**: Dual-mode Selective State Space Block executing either compiled `mamba-ssm` CUDA operations or pure PyTorch recurrence fallback.
3. **`BiMambaSequenceModel`**: 2-layer sequence backbone executing forward and reverse scans with masked sequence flipping and residual skip connections.
4. **`GatedTemporalAttentionPooling`**: Parametric temporal pooling $\alpha_t = \text{softmax}(w^T (\tanh(W_a h_t) \odot \sigma(W_b h_t)))$ with $-\infty$ pre-softmax masking over padded frames.
5. **`ClassificationHead`**: `LayerNorm(128)` -> `Dropout(0.4)` -> `Linear(128, 2)`.

In [ ]:
class Spatial3DCNNEncoder(nn.Module):
    """4-block volumetric 3D-CNN spatial encoder extracting 128-d latent representations."""
    def __init__(
        self,
        in_channels: int = 1,
        out_features: int = 128,
        leaky_relu_slope: float = 0.01,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.in_channels = in_channels
        self.out_features = out_features
        self.block1 = self._make_block(in_channels, 32, leaky_relu_slope)
        self.block2 = self._make_block(32, 64, leaky_relu_slope)
        self.block3 = self._make_block(64, 128, leaky_relu_slope)
        self.block4 = self._make_block(128, 256, leaky_relu_slope)
        self.global_pool = nn.AdaptiveAvgPool3d((1, 1, 1))
        self.dropout = nn.Dropout(p=dropout) if dropout > 0.0 else nn.Identity()
        self.fc = nn.Linear(256, out_features)

    @staticmethod
    def _make_block(in_c: int, out_c: int, slope: float) -> nn.Sequential:
        return nn.Sequential(
            nn.Conv3d(in_c, out_c, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm3d(out_c),
            nn.LeakyReLU(negative_slope=slope, inplace=True),
            nn.MaxPool3d(kernel_size=2, stride=2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.dim() == 5:
            is_sequence = False
            batch_size, time_steps = x.size(0), 1
            x_flat = x
        elif x.dim() == 6:
            is_sequence = True
            batch_size, time_steps, c, h, w, d = x.shape
            x_flat = x.reshape(batch_size * time_steps, c, h, w, d)
        else:
            raise ValueError(f'Expected 5D or 6D tensor, got shape {tuple(x.shape)}')

        h1 = self.block1(x_flat)
        h2 = self.block2(h1)
        h3 = self.block3(h2)
        h4 = self.block4(h3)
        pooled = self.global_pool(h4).flatten(1)
        dropped = self.dropout(pooled)
        out = self.fc(dropped)
        if is_sequence:
            return out.reshape(batch_size, time_steps, self.out_features)
        return out

def selective_scan_fn(
    u: torch.Tensor,
    delta: torch.Tensor,
    A: torch.Tensor,
    B: torch.Tensor,
    C: torch.Tensor,
    D: Optional[torch.Tensor] = None,
    z: Optional[torch.Tensor] = None,
    delta_bias: Optional[torch.Tensor] = None,
    delta_softplus: bool = False,
) -> torch.Tensor:
    """Pure PyTorch selective scan reference implementation."""
    if delta_bias is not None:
        delta = delta + delta_bias.unsqueeze(-1)
    if delta_softplus:
        delta = F.softplus(delta)

    batch_size, dim, seq_len = u.shape
    d_state = A.shape[1]
    state = torch.zeros(batch_size, dim, d_state, device=u.device, dtype=u.dtype)

    ys = []
    for t in range(seq_len):
        u_t = u[:, :, t]
        delta_t = delta[:, :, t]
        b_t = B[:, :, t]
        c_t = C[:, :, t]

        a_bar = torch.exp(delta_t.unsqueeze(-1) * A.unsqueeze(0))
        b_bar = delta_t.unsqueeze(-1) * b_t.unsqueeze(1)
        state = a_bar * state + b_bar * u_t.unsqueeze(-1)
        y_t = torch.sum(state * c_t.unsqueeze(1), dim=-1)
        if D is not None:
            y_t = y_t + u_t * D.unsqueeze(0)
        ys.append(y_t)

    y = torch.stack(ys, dim=-1)
    if z is not None:
        y = y * F.silu(z)
    return y

class MambaBlock(nn.Module):
    """Selective state space Mamba block with dual-mode CUDA/PyTorch fallback."""
    def __init__(
        self,
        d_model: int = 128,
        d_state: int = 16,
        d_conv: int = 4,
        expand: int = 2,
        dt_rank: Optional[int] = None,
        force_pure_pytorch: bool = False,
    ) -> None:
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.expand = expand
        self.d_inner = expand * d_model
        self.dt_rank = dt_rank if dt_rank is not None else math.ceil(d_model / 16)
        self.force_pure_pytorch = force_pure_pytorch

        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False)
        self.conv1d = nn.Conv1d(self.d_inner, self.d_inner, kernel_size=d_conv, bias=True, groups=self.d_inner, padding=d_conv - 1)
        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + 2 * self.d_state, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)

        a_log = torch.log(torch.arange(1, self.d_state + 1, dtype=torch.float32)).unsqueeze(0).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(a_log)
        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, _ = x.shape
        xz = self.in_proj(x)
        x_branch, z = xz.chunk(2, dim=-1)
        x_conv = self.conv1d(x_branch.transpose(1, 2))[..., :seq_len]
        x_act = F.silu(x_conv)

        x_act_t = x_act.transpose(1, 2)
        ssm_params = self.x_proj(x_act_t)
        delta, b_proj, c_proj = torch.split(ssm_params, [self.dt_rank, self.d_state, self.d_state], dim=-1)
        delta = self.dt_proj(delta).transpose(1, 2)
        b_proj = b_proj.transpose(1, 2)
        c_proj = c_proj.transpose(1, 2)
        z_t = z.transpose(1, 2)
        A = -torch.exp(self.A_log)

        can_use_cuda = (HAS_CUDA_MAMBA and not self.force_pure_pytorch and x.is_cuda and cuda_selective_scan_fn is not None)
        if can_use_cuda:
            y = cuda_selective_scan_fn(x_act, delta, A, b_proj, c_proj, self.D, z=z_t, delta_bias=None, delta_softplus=True)
        else:
            y = selective_scan_fn(u=x_act, delta=delta, A=A, B=b_proj, C=c_proj, D=self.D, z=z_t, delta_bias=None, delta_softplus=True)
        return self.out_proj(y.transpose(1, 2))

class BiMambaSequenceModel(nn.Module):
    """2-layer Bidirectional Mamba sequence model with pure PyTorch fallback."""
    def __init__(
        self,
        d_model: int = 128,
        d_state: int = 16,
        d_conv: int = 4,
        expand: int = 2,
        num_layers: int = 2,
        dropout: float = 0.1,
        force_pure_pytorch: bool = False,
    ) -> None:
        super().__init__()
        self.d_model = d_model
        self.num_layers = num_layers
        self.fwd_layers = nn.ModuleList([MambaBlock(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand, force_pure_pytorch=force_pure_pytorch) for _ in range(num_layers)])
        self.rev_layers = nn.ModuleList([MambaBlock(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand, force_pure_pytorch=force_pure_pytorch) for _ in range(num_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(num_layers)])
        self.dropouts = nn.ModuleList([nn.Dropout(p=dropout) if dropout > 0.0 else nn.Identity() for _ in range(num_layers)])
        self.final_norm = nn.LayerNorm(d_model)

    @staticmethod
    def _flip_sequence(tensor: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        if mask is None:
            return torch.flip(tensor, dims=[1])
        batch_size, seq_len, _ = tensor.shape
        flipped = torch.zeros_like(tensor)
        valid_lengths = mask.sum(dim=1).long()
        for b in range(batch_size):
            length = valid_lengths[b].item()
            if length > 0:
                flipped[b, :length] = torch.flip(tensor[b, :length], dims=[0])
        return flipped

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        out = x
        for fwd_block, rev_block, norm, dropout in zip(self.fwd_layers, self.rev_layers, self.norms, self.dropouts):
            normed = norm(out)
            if mask is not None:
                normed = normed.masked_fill(~mask.unsqueeze(-1), 0.0)
            fwd_out = fwd_block(normed)
            if mask is not None:
                fwd_out = fwd_out.masked_fill(~mask.unsqueeze(-1), 0.0)
            normed_rev = self._flip_sequence(normed, mask)
            rev_out_flipped = rev_block(normed_rev)
            rev_out = self._flip_sequence(rev_out_flipped, mask)
            if mask is not None:
                rev_out = rev_out.masked_fill(~mask.unsqueeze(-1), 0.0)
            layer_out = dropout(fwd_out + rev_out)
            out = out + layer_out
            if mask is not None:
                out = out.masked_fill(~mask.unsqueeze(-1), 0.0)
        out = self.final_norm(out)
        if mask is not None:
            out = out.masked_fill(~mask.unsqueeze(-1), 0.0)
        return out

class GatedTemporalAttentionPooling(nn.Module):
    """Gated Temporal Attention Pooling with -inf masking over padded frames."""
    def __init__(self, in_features: int = 128, latent_dim: int = 64, dropout: float = 0.0) -> None:
        super().__init__()
        self.linear_a = nn.Linear(in_features, latent_dim)
        self.linear_b = nn.Linear(in_features, latent_dim)
        self.linear_w = nn.Linear(latent_dim, 1, bias=False)
        self.dropout = nn.Dropout(p=dropout) if dropout > 0.0 else nn.Identity()

    def forward(self, h: torch.Tensor, mask: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        if h.dim() == 2:
            unbatched = True
            h = h.unsqueeze(0)
            if mask is not None:
                mask = mask.unsqueeze(0)
        elif h.dim() == 3:
            unbatched = False
        else:
            raise ValueError(f'Expected 2D or 3D tensor, got shape {tuple(h.shape)}')

        a_out = torch.tanh(self.linear_a(h))
        b_out = torch.sigmoid(self.linear_b(h))
        gated = self.dropout(a_out * b_out)
        scores = self.linear_w(gated).squeeze(-1)
        if mask is not None:
            scores = scores.masked_fill(~mask.bool(), float('-inf'))
        attention_weights = F.softmax(scores, dim=-1)
        if mask is not None:
            attention_weights = torch.nan_to_num(attention_weights, nan=0.0)
        pooled = torch.sum(attention_weights.unsqueeze(-1) * h, dim=1)
        if unbatched:
            return pooled.squeeze(0), attention_weights.squeeze(0)
        return pooled, attention_weights

class ClassificationHead(nn.Module):
    """Classification head: LayerNorm -> Dropout(0.4) -> Linear(128, 2)."""
    def __init__(self, in_features: int = 128, num_classes: int = 2, dropout: float = 0.4) -> None:
        super().__init__()
        self.norm = nn.LayerNorm(in_features)
        self.dropout = nn.Dropout(p=dropout) if dropout > 0.0 else nn.Identity()
        self.fc = nn.Linear(in_features, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(self.dropout(self.norm(x)))

class SpatioTemporalBiMamba(nn.Module):
    """Integrated Spatio-Temporal 3D-CNN with Bidirectional Mamba architecture."""
    def __init__(
        self,
        in_channels: int = 1,
        d_model: int = 128,
        d_state: int = 16,
        d_conv: int = 4,
        expand: int = 2,
        num_layers: int = 2,
        num_classes: int = 2,
        spatial_dropout: float = 0.1,
        temporal_dropout: float = 0.1,
        attention_dropout: float = 0.0,
        head_dropout: float = 0.4,
        force_pure_pytorch: bool = False,
    ) -> None:
        super().__init__()
        self.spatial_encoder = Spatial3DCNNEncoder(in_channels=in_channels, out_features=d_model, dropout=spatial_dropout)
        self.sequence_model = BiMambaSequenceModel(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand, num_layers=num_layers, dropout=temporal_dropout, force_pure_pytorch=force_pure_pytorch)
        self.attention_pooling = GatedTemporalAttentionPooling(in_features=d_model, latent_dim=d_model // 2 if d_model >= 2 else d_model, dropout=attention_dropout)
        self.classification_head = ClassificationHead(in_features=d_model, num_classes=num_classes, dropout=head_dropout)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        spatial_features = self.spatial_encoder(x)
        h_seq = self.sequence_model(spatial_features, mask=mask)
        pooled, attention_weights = self.attention_pooling(h_seq, mask=mask)
        logits = self.classification_head(pooled)
        return logits, attention_weights

## 6. Focal Loss, Optimization, and Early Stopping Checkpointer

We employ Focal Loss with $\alpha=0.25, \gamma=2.0$ modulated by fold-level inverse frequency class weights to mitigate diagnostic class imbalances.
The model is optimized using AdamW ($lr=1e-4, \text{weight decay}=1e-4$) with Cosine Annealing learning rate schedule and validation loss early stopping monitoring.

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss for binary and multi-class classification with hard sample focusing."""
    def __init__(self, alpha: Optional[Union[float, Sequence[float], torch.Tensor]] = 0.25, gamma: float = 2.0, reduction: str = 'mean') -> None:
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction
        if alpha is not None and not isinstance(alpha, torch.Tensor):
            self.alpha = torch.tensor(alpha, dtype=torch.float32)
        else:
            self.alpha = alpha

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_term = (1.0 - pt) ** self.gamma
        if self.alpha is not None:
            alpha_t = self.alpha.to(inputs.device)[targets]
            loss = alpha_t * focal_term * ce_loss
        else:
            loss = focal_term * ce_loss
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss

def compute_fold_class_weights(labels: Union[Sequence[int], np.ndarray, torch.Tensor], alpha: Optional[float] = None) -> torch.Tensor:
    """Calculate inverse frequency class weights from training fold label distribution."""
    arr = labels.detach().cpu().numpy().ravel() if isinstance(labels, torch.Tensor) else np.asarray(labels).ravel()
    n_samples = len(arr)
    if n_samples == 0:
        return torch.tensor([1.0, 1.0], dtype=torch.float32)
    c0 = max(int(np.sum(arr == 0)), 1)
    c1 = max(int(np.sum(arr == 1)), 1)
    if alpha is not None:
        raw_w0 = (1.0 - alpha) / float(c0)
        raw_w1 = alpha / float(c1)
        tot = raw_w0 + raw_w1
        return torch.tensor([2.0 * raw_w0 / tot, 2.0 * raw_w1 / tot], dtype=torch.float32)
    return torch.tensor([float(n_samples) / (2.0 * float(c0)), float(n_samples) / (2.0 * float(c1))], dtype=torch.float32)

class SpatiotemporalEarlyStopping:
    """Early stopping monitor tracking validation loss and saving top clinical checkpoints."""
    def __init__(self, patience: int = 15, min_delta: float = 1e-4, checkpoint_path: Optional[Union[str, Path]] = None) -> None:
        self.patience = patience
        self.min_delta = min_delta
        self.checkpoint_path = Path(checkpoint_path) if checkpoint_path is not None else None
        self.counter = 0
        self.best_loss = float('inf')
        self.best_score = float('-inf')
        self.early_stop = False

    def __call__(self, val_loss: float, val_metrics: Dict[str, float], model: Optional[nn.Module] = None) -> bool:
        loss_val = float(val_loss)
        bal_acc = float(val_metrics.get('balanced_accuracy', 0.0))
        roc_auc = float(val_metrics.get('roc_auc', 0.0))
        if math.isnan(roc_auc):
            roc_auc = bal_acc
        composite_score = 0.5 * (bal_acc + roc_auc)

        if loss_val < (self.best_loss - self.min_delta):
            self.best_loss = loss_val
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

        if composite_score > self.best_score:
            self.best_score = composite_score
            if self.checkpoint_path is not None and model is not None:
                self.checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
                core_model = model.module if hasattr(model, 'module') else model
                torch.save(core_model.state_dict(), self.checkpoint_path)
        return self.early_stop

## 7. Joint Site-and-Class Stratified Cross-Validation Partitioning

To guarantee unbiased evaluation across multi-institution rs-fMRI data, every cross-validation fold must preserve identical diagnostic ratios (ASD vs. TC) and institutional representation across all 20 ABIDE acquisition sites.
Our joint stratifier partitions subjects by `(site_id, label)` strata without inter-fold sample leakage.

In [ ]:
def get_site_and_class_stratified_folds(
    df: pd.DataFrame,
    n_splits: int = 5,
    random_state: int = 42,
    site_col: str = 'site_id',
    label_col: str = 'label',
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """Generate joint Site-and-Class Stratified K-Fold index splits."""
    if n_splits < 2:
        raise ValueError(f'n_splits must be at least 2, got {n_splits}')
    n_samples = len(df)
    if n_samples < n_splits:
        raise ValueError(f'Cannot partition {n_samples} samples into {n_splits} folds.')

    rng = np.random.RandomState(random_state)
    fold_assignments = np.full(n_samples, -1, dtype=int)
    groups = df.groupby([site_col, label_col], sort=True).indices

    remainder_offset = 0
    for _, indices in groups.items():
        idx_arr = np.array(indices, copy=True)
        rng.shuffle(idx_arr)
        n_items = len(idx_arr)
        n_full = (n_items // n_splits) * n_splits
        if n_full > 0:
            fold_assignments[idx_arr[:n_full]] = np.tile(np.arange(n_splits), n_full // n_splits)
        remainder = n_items % n_splits
        if remainder > 0:
            rem_folds = [(remainder_offset + i) % n_splits for i in range(remainder)]
            fold_assignments[idx_arr[n_full:]] = rem_folds
            remainder_offset = (remainder_offset + remainder) % n_splits

    splits: List[Tuple[np.ndarray, np.ndarray]] = []
    all_indices = np.arange(n_samples)
    for fold_idx in range(n_splits):
        val_idx = all_indices[fold_assignments == fold_idx]
        train_idx = all_indices[fold_assignments != fold_idx]
        splits.append((train_idx, val_idx))
    return splits

## 8. Training and Evaluation Functions with BF16 Mixed Precision

We construct training and validation epoch runners incorporating:
- `torch.amp.autocast(dtype=torch.bfloat16)` for high-throughput native BF16 tensor execution.
- Gradient clipping (`max_norm=1.0`) to stabilize deep sequence state-space updates.
- Extraction of temporal attention weights $\alpha_t$ from validation forward passes.

In [ ]:
def train_spatiotemporal_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
    use_bf16: bool = True,
    gradient_clip_norm: float = 1.0,
) -> float:
    """Execute one training epoch using BF16 autocast and gradient clipping."""
    model.train()
    total_loss = 0.0
    total_samples = 0
    optimizer.zero_grad()
    device_type = 'cuda' if device.type == 'cuda' else 'cpu'

    for batch in dataloader:
        volumes = batch['volume'].to(device, non_blocking=True)
        masks = batch['mask'].to(device, non_blocking=True)
        targets = batch['label'].to(device, non_blocking=True)
        batch_size = targets.size(0)

        with torch.amp.autocast(device_type=device_type, dtype=torch.bfloat16, enabled=use_bf16):
            outputs = model(volumes, mask=masks)
            logits = outputs[0] if isinstance(outputs, tuple) else outputs

        loss = criterion(logits.float(), targets)
        loss.backward()
        if gradient_clip_norm > 0.0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=gradient_clip_norm)
        optimizer.step()
        optimizer.zero_grad()

        total_loss += float(loss.item()) * batch_size
        total_samples += batch_size

    return total_loss / max(total_samples, 1)

def evaluate_spatiotemporal_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    use_bf16: bool = True,
    return_attention: bool = True,
) -> Tuple[float, Dict[str, float], np.ndarray, np.ndarray, np.ndarray, np.ndarray, Optional[np.ndarray]]:
    """Execute evaluation epoch returning loss, clinical metrics, predictions, attention weights, and masks."""
    model.eval()
    total_loss = 0.0
    total_samples = 0
    all_targets: List[np.ndarray] = []
    all_preds: List[np.ndarray] = []
    all_probs: List[np.ndarray] = []
    all_attns: List[np.ndarray] = []
    all_masks: List[np.ndarray] = []
    device_type = 'cuda' if device.type == 'cuda' else 'cpu'

    with torch.no_grad():
        for batch in dataloader:
            volumes = batch['volume'].to(device, non_blocking=True)
            masks = batch['mask'].to(device, non_blocking=True)
            targets = batch['label'].to(device, non_blocking=True)
            batch_size = targets.size(0)

            with torch.amp.autocast(device_type=device_type, dtype=torch.bfloat16, enabled=use_bf16):
                outputs = model(volumes, mask=masks)
                logits = outputs[0] if isinstance(outputs, tuple) else outputs
                attn = outputs[1] if isinstance(outputs, tuple) and len(outputs) > 1 else None

            loss = criterion(logits.float(), targets)
            probs = torch.softmax(logits.float(), dim=-1)
            preds = torch.argmax(logits, dim=-1)

            total_loss += float(loss.item()) * batch_size
            total_samples += batch_size
            all_targets.append(targets.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
            if attn is not None:
                all_attns.append(attn.detach().cpu().float().numpy())
            if masks is not None:
                all_masks.append(masks.detach().cpu().bool().numpy())

    y_true = np.concatenate(all_targets) if all_targets else np.array([], dtype=int)
    y_pred = np.concatenate(all_preds) if all_preds else np.array([], dtype=int)
    y_prob = np.concatenate(all_probs) if all_probs else np.empty((0, 2), dtype=float)
    y_attn = np.concatenate(all_attns, axis=0) if all_attns else np.empty((0, 0), dtype=float)
    y_mask = np.concatenate(all_masks, axis=0) if all_masks else None
    avg_loss = total_loss / max(total_samples, 1)

    # Compute clinical metrics
    acc = float(accuracy_score(y_true, y_pred)) if len(y_true) > 0 else 0.0
    bal_acc = float(balanced_accuracy_score(y_true, y_pred)) if len(y_true) > 0 else 0.0
    try:
        auc = float(roc_auc_score(y_true, y_prob[:, 1])) if len(np.unique(y_true)) > 1 else 0.5
    except Exception:
        auc = 0.5
    sens = float(recall_score(y_true, y_pred, pos_label=1, zero_division=0)) if len(y_true) > 0 else 0.0
    spec = float(recall_score(y_true, y_pred, pos_label=0, zero_division=0)) if len(y_true) > 0 else 0.0
    f1 = float(f1_score(y_true, y_pred, zero_division=0)) if len(y_true) > 0 else 0.0

    metrics = {
        'loss': avg_loss,
        'accuracy': acc,
        'balanced_accuracy': bal_acc,
        'roc_auc': auc,
        'sensitivity': sens,
        'recall': sens,
        'specificity': spec,
        'f1_score': f1,
    }
    return avg_loss, metrics, y_true, y_pred, y_prob, y_attn, y_mask

## 9. 5-Fold Cross-Validation Orchestrator (`run_spatiotemporal_kfold_cv`)

Executes end-to-end 5-fold cross-validation, maintaining fold model isolation, weighted Focal Loss per fold, cosine learning rate scheduling, validation early stopping, best weight checkpointing, and out-of-fold prediction and attention aggregation.

In [ ]:
def run_spatiotemporal_kfold_cv(
    manifest: pd.DataFrame,
    dataset: Dataset,
    n_splits: int = 5,
    epochs: int = 30,
    batch_size: int = 2,
    learning_rate: float = 1e-4,
    weight_decay: float = 1e-4,
    gradient_clip_norm: float = 1.0,
    early_stopping_patience: int = 15,
    checkpoint_dir: Optional[Union[str, Path]] = None,
    device: Optional[Union[str, torch.device]] = None,
    use_bf16: bool = True,
    use_data_parallel: bool = True,
    random_state: int = 42,
) -> Dict[str, Any]:
    """Execute Site-and-Class Stratified 5-Fold Cross-Validation for SpatioTemporalBiMamba."""
    exec_device = torch.device(device) if device is not None else torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    ckpt_dir = Path(checkpoint_dir) if checkpoint_dir is not None else Path('/kaggle/working/artifacts/models/spatiotemporal')
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    folds = get_site_and_class_stratified_folds(manifest, n_splits=n_splits, random_state=random_state)

    fold_results: List[Dict[str, Any]] = []
    oof_targets: List[int] = []
    oof_preds: List[int] = []
    oof_probs: List[List[float]] = []
    oof_attentions: List[np.ndarray] = []
    oof_masks: List[np.ndarray] = []

    for fold_idx, (train_idx, val_idx) in enumerate(folds, 1):
        print(f'\n>>> Starting Fold {fold_idx}/{n_splits} (Train: {len(train_idx)}, Val: {len(val_idx)}) <<<')
        train_labels = manifest.iloc[train_idx]['label'].to_numpy()
        class_weights = compute_fold_class_weights(train_labels, alpha=0.25).to(exec_device)
        criterion = FocalLoss(alpha=class_weights, gamma=2.0)

        model = SpatioTemporalBiMamba().to(exec_device)
        parallel_model = nn.DataParallel(model) if (use_data_parallel and torch.cuda.device_count() > 1) else model

        optimizer = torch.optim.AdamW(parallel_model.parameters(), lr=learning_rate, weight_decay=weight_decay)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

        fold_ckpt = ckpt_dir / f'best_spatiotemporal_fold_{fold_idx}.pt'
        early_stopping = SpatiotemporalEarlyStopping(patience=early_stopping_patience, checkpoint_path=fold_ckpt)

        train_sub = torch.utils.data.Subset(dataset, train_idx.tolist())
        val_sub = torch.utils.data.Subset(dataset, val_idx.tolist())

        train_loader = DataLoader(train_sub, batch_size=batch_size, shuffle=True, collate_fn=spatiotemporal_collate_fn)
        val_loader = DataLoader(val_sub, batch_size=batch_size, shuffle=False, collate_fn=spatiotemporal_collate_fn)

        for epoch in range(1, epochs + 1):
            t_loss = train_spatiotemporal_epoch(
                model=parallel_model, dataloader=train_loader, optimizer=optimizer,
                criterion=criterion, device=exec_device, use_bf16=use_bf16, gradient_clip_norm=gradient_clip_norm,
            )
            v_loss, v_metrics, _, _, _, _, _ = evaluate_spatiotemporal_epoch(
                model=parallel_model, dataloader=val_loader, criterion=criterion,
                device=exec_device, use_bf16=use_bf16, return_attention=False,
            )
            scheduler.step()
            if epoch % 5 == 0 or epoch == 1:
                print(f'  [Fold {fold_idx} | Epoch {epoch:2d}/{epochs}] Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f} | Bal Acc: {v_metrics["balanced_accuracy"]*100:.2f}% | AUC: {v_metrics["roc_auc"]:.3f}')
            if early_stopping(val_loss=v_loss, val_metrics=v_metrics, model=parallel_model):
                print(f'  [Fold {fold_idx}] Early stopping triggered at epoch {epoch}.')
                break

        if fold_ckpt.exists():
            core_m = parallel_model.module if hasattr(parallel_model, 'module') else parallel_model
            core_m.load_state_dict(torch.load(fold_ckpt, map_location='cpu', weights_only=False))
            core_m.to(exec_device)

        _, final_m, y_t_f, y_p_f, y_pr_f, y_att_f, y_mask_f = evaluate_spatiotemporal_epoch(
            model=parallel_model, dataloader=val_loader, criterion=criterion,
            device=exec_device, use_bf16=use_bf16, return_attention=True,
        )
        fold_results.append({
            'fold': fold_idx,
            **final_m,
            'metrics': final_m,
            'y_true': y_t_f,
            'y_pred': y_p_f,
            'y_prob': y_pr_f,
            'y_attn': y_att_f,
            'y_mask': y_mask_f,
        })
        oof_targets.extend(y_t_f.tolist())
        oof_preds.extend(y_p_f.tolist())
        oof_probs.extend(y_pr_f.tolist())
        if len(y_att_f) > 0:
            oof_attentions.append(y_att_f)
        if y_mask_f is not None:
            oof_masks.append(y_mask_f)

    oof_targets_np = np.array(oof_targets)
    oof_preds_np = np.array(oof_preds)
    oof_probs_np = np.array(oof_probs)
    oof_attns_np = np.concatenate(oof_attentions, axis=0) if oof_attentions else np.empty((0, 0))
    oof_masks_np = np.concatenate(oof_masks, axis=0) if oof_masks else None

    return {
        'fold_results': fold_results,
        'oof_targets': oof_targets_np,
        'oof_preds': oof_preds_np,
        'oof_probs': oof_probs_np,
        'oof_attentions': oof_attns_np,
        'oof_masks': oof_masks_np,
    }

## 10. Out-of-Fold Validation Prediction Aggregation & Aggregate ROC Curve Visualization

We aggregate out-of-fold validation predictions across all 5 folds and plot cross-fold and aggregate ROC curves with computed AUC.

In [ ]:
def plot_spatiotemporal_roc_curve(
    y_true: Union[Sequence[int], np.ndarray],
    y_prob: Union[Sequence[float], np.ndarray],
    fold_evaluations: Optional[List[Dict[str, Any]]] = None,
    save_path: Optional[Union[str, Path]] = '/kaggle/working/spatiotemporal_roc_curve.png',
    title: str = 'Out-of-Fold Cross-Validation ROC Curve (3D-CNN + Bi-Mamba)',
) -> Tuple[Any, Any]:
    """Plot aggregate out-of-fold ROC curve with calculated AUC."""
    y_true_arr = np.asarray(y_true).ravel()
    y_prob_arr = np.asarray(y_prob)
    y_score = y_prob_arr[:, 1] if y_prob_arr.ndim == 2 and y_prob_arr.shape[1] == 2 else y_prob_arr.ravel()

    fig, ax = plt.subplots(figsize=(8, 6), dpi=150)
    if fold_evaluations:
        for fold_idx, fold_eval in enumerate(fold_evaluations, 1):
            if 'y_true' in fold_eval and 'y_prob' in fold_eval:
                f_true = np.asarray(fold_eval['y_true']).ravel()
                f_prob = np.asarray(fold_eval['y_prob'])
                f_score = f_prob[:, 1] if f_prob.ndim == 2 and f_prob.shape[1] == 2 else f_prob.ravel()
                if len(np.unique(f_true)) > 1:
                    f_fpr, f_tpr, _ = roc_curve(f_true, f_score)
                    f_auc = roc_auc_score(f_true, f_score)
                    ax.plot(f_fpr, f_tpr, linestyle='--', alpha=0.4, label=f'Fold {fold_idx} (AUC = {f_auc:.3f})')

    fpr, tpr, _ = roc_curve(y_true_arr, y_score)
    overall_auc = roc_auc_score(y_true_arr, y_score)
    ax.plot(fpr, tpr, color='#1f77b4', lw=2.5, label=f'Aggregate Out-of-Fold (AUC = {overall_auc:.3f})')
    ax.plot([0, 1], [0, 1], linestyle=':', color='gray', lw=1.5, label='Chance Level (AUC = 0.500)')

    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
    ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(loc='lower right', frameon=True, fontsize=10)
    ax.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()

    if save_path is not None:
        p = Path(save_path)
        p.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(p, dpi=150, bbox_inches='tight')
        print(f'[INFO] Saved aggregate ROC curve to: {p}')
    return fig, ax

## 11. Dynamic Temporal Attention Trajectory Extraction & Interpretation (ASD vs. TC)

We extract frame-by-frame attention weights $\alpha_t$ from Gated Temporal Attention Pooling, compute cohort means and standard error bands (SEM), and highlight key temporal intervals where hemodynamic attention discriminates between ASD and TC cohorts.

In [ ]:
def plot_temporal_attention_trajectories(
    attention_weights: np.ndarray,
    labels: Union[Sequence[int], np.ndarray],
    masks: Optional[np.ndarray] = None,
    tr_seconds: float = 2.0,
    save_path: Optional[Union[str, Path]] = '/kaggle/working/temporal_attention_trajectories.png',
    title: str = 'Mean Dynamic Temporal Attention Trajectories: ASD vs. TC Cohorts',
) -> Tuple[Any, Any]:
    """Plot mean temporal attention trajectories contrasting ASD vs. TC cohorts across scan duration."""
    attentions = np.asarray(attention_weights, dtype=np.float32)
    target_labels = np.asarray(labels).ravel()
    if attentions.ndim != 2:
        raise ValueError(f'Expected 2D array (N, T) for attention_weights, got shape {attentions.shape}')

    _, seq_len = attentions.shape
    time_points = np.arange(1, seq_len + 1)
    time_seconds = time_points * tr_seconds

    asd_mask = (target_labels == 1)
    tc_mask = (target_labels == 0)

    def _compute_cohort_stats(cohort_bool_mask: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        cohort_attns = attentions[cohort_bool_mask]
        if len(cohort_attns) == 0:
            return np.zeros(seq_len), np.zeros(seq_len)
        if masks is not None:
            cohort_valid = masks[cohort_bool_mask]
            means, sems = [], []
            for t in range(seq_len):
                valid_vals = cohort_attns[cohort_valid[:, t], t]
                if len(valid_vals) > 0:
                    mean_val = float(np.mean(valid_vals))
                    sem_val = float(np.std(valid_vals) / np.sqrt(len(valid_vals))) if len(valid_vals) > 1 else 0.0
                else:
                    mean_val, sem_val = 0.0, 0.0
                means.append(mean_val)
                sems.append(sem_val)
            return np.array(means), np.array(sems)
        means = np.mean(cohort_attns, axis=0)
        n_sub = len(cohort_attns)
        sems = np.std(cohort_attns, axis=0) / np.sqrt(n_sub) if n_sub > 1 else np.zeros(seq_len)
        return means, sems

    asd_mean, asd_sem = _compute_cohort_stats(asd_mask)
    tc_mean, tc_sem = _compute_cohort_stats(tc_mask)
    divergence = np.abs(asd_mean - tc_mean)

    fig, (ax_main, ax_diff) = plt.subplots(
        nrows=2, ncols=1, figsize=(12, 8), dpi=150,
        gridspec_kw={'height_ratios': [2.5, 1]}, sharex=True,
    )

    n_asd = int(np.sum(asd_mask))
    n_tc = int(np.sum(tc_mask))
    ax_main.plot(time_seconds, asd_mean, color='#d62728', lw=2.0, label=f'ASD Cohort (N={n_asd})')
    ax_main.fill_between(time_seconds, asd_mean - asd_sem, asd_mean + asd_sem, color='#d62728', alpha=0.2)
    ax_main.plot(time_seconds, tc_mean, color='#1f77b4', lw=2.0, label=f'TC Cohort (N={n_tc})')
    ax_main.fill_between(time_seconds, tc_mean - tc_sem, tc_mean + tc_sem, color='#1f77b4', alpha=0.2)

    ax_main.set_ylabel('Temporal Attention Weight (alpha_t)', fontsize=11)
    ax_main.set_title(title, fontsize=13, fontweight='bold')
    ax_main.legend(loc='upper right', frameon=True, fontsize=10)
    ax_main.grid(True, linestyle='--', alpha=0.5)

    ax_diff.plot(time_seconds, divergence, color='#2ca02c', lw=1.8, label='|ASD - TC| Attention Differential')
    ax_diff.fill_between(time_seconds, 0, divergence, color='#2ca02c', alpha=0.2)

    if len(divergence) > 0 and np.max(divergence) > 0:
        peak_idx = int(np.argmax(divergence))
        peak_t = time_seconds[peak_idx]
        peak_val = divergence[peak_idx]
        ax_diff.annotate(
            f'Peak Divergence (t={peak_t:.0f}s, frame={peak_idx+1})',
            xy=(peak_t, peak_val),
            xytext=(peak_t + 10, peak_val * 0.9 + 1e-4),
            arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=6),
            fontsize=9,
        )

    ax_diff.set_xlabel('Scan Acquisition Time (Seconds, TR=2.0s)', fontsize=11)
    ax_diff.set_ylabel('Attention Divergence', fontsize=10)
    ax_diff.legend(loc='upper right', frameon=True, fontsize=9)
    ax_diff.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()

    if save_path is not None:
        p = Path(save_path)
        p.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(p, dpi=150, bbox_inches='tight')
        print(f'[INFO] Saved temporal attention trajectories to: {p}')
    return fig, (ax_main, ax_diff)

## 12. End-to-End Execution and Results Summary

We discover all mounted dataset partitions, configure the 5-fold cross-validation loop, train the Spatio-Temporal Bi-Mamba model across visible GPUs with native BF16, aggregate out-of-fold predictions, and render the interpretability figures.

In [ ]:
# Hyperparameters
EPOCHS = 30
BATCH_SIZE = max(4, num_gpus) if num_gpus > 0 else 2
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
GRADIENT_CLIP_NORM = 1.0
EARLY_STOPPING_PATIENCE = 15
N_SPLITS = 5
CHECKPOINT_DIR = Path('/kaggle/working/artifacts/models/spatiotemporal')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Step 1: Auto-discover mounted dataset parts across /kaggle/input/
print('=' * 80)
print('STEP 1: MULTI-PART DATASET AUTO-DISCOVERY')
print('=' * 80)
manifest_df, file_mapping = discover_multisite_dataset()
print(f'[SUCCESS] Discovered {len(manifest_df)} total subjects across mounted partitions.')

# Breakdown by detected parts and sites
print('\nCohort Partition Breakdown:')
for part_num in sorted(PART_SITES.keys()):
    sub_part = manifest_df[manifest_df['part_num'] == part_num]
    sites = PART_SITES[part_num]
    print(f'  Part {part_num} (Sites: {sites}): {len(sub_part)} subjects on disk')

n_asd = int((manifest_df['label'] == 1).sum())
n_tc = int((manifest_df['label'] == 0).sum())
print(f'Diagnostic Cohort Balance: ASD={n_asd} | TC={n_tc}')
print('=' * 80)

# Step 2: Initialize full SpatioTemporalFMRIDataset
full_dataset = SpatioTemporalFMRIDataset(
    manifest=manifest_df,
    target_spatial_shape=(64, 64, 36),
    target_seq_len=200,
    trim_start=10,
    trim_end=10,
    normalize=True,
)

# Step 3: Run Joint Site-and-Class Stratified 5-Fold Cross-Validation
print('\n' + '=' * 80)
print('STEP 2: JOINT SITE-AND-CLASS STRATIFIED 5-FOLD CROSS-VALIDATION')
print('=' * 80)
cv_results = run_spatiotemporal_kfold_cv(
    manifest=manifest_df,
    dataset=full_dataset,
    n_splits=N_SPLITS,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    gradient_clip_norm=GRADIENT_CLIP_NORM,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    checkpoint_dir=CHECKPOINT_DIR,
    device=device,
    use_bf16=USE_BF16,
    use_data_parallel=USE_DATA_PARALLEL,
)

fold_results = cv_results['fold_results']
oof_targets_np = cv_results['oof_targets']
oof_preds_np = cv_results['oof_preds']
oof_probs_np = cv_results['oof_probs']
oof_attns_np = cv_results['oof_attentions']
oof_masks_np = cv_results.get('oof_masks', None)

print('\n' + '=' * 80)
print('CROSS-VALIDATION CLINICAL RESULTS SUMMARY')
print('=' * 80)
metric_names = ['balanced_accuracy', 'roc_auc', 'sensitivity', 'specificity', 'f1_score', 'loss']
for m in metric_names:
    vals = [fr['metrics'][m] for fr in fold_results if m in fr['metrics'] and not math.isnan(fr['metrics'][m])]
    if vals:
        print(f'  {m.replace("_", " ").title():<22}: {np.mean(vals)*100:6.2f}% +/- {np.std(vals)*100:5.2f}%' if m != 'loss' else f'  {m.title():<22}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}')

# Step 4: Render ROC Curves & Temporal Attention Visualizations
print('\n' + '=' * 80)
print('STEP 3: RENDERING CLINICAL INTERPRETABILITY VISUALIZATIONS')
print('=' * 80)
plot_spatiotemporal_roc_curve(
    y_true=oof_targets_np,
    y_prob=oof_probs_np,
    fold_evaluations=fold_results,
    save_path='/kaggle/working/spatiotemporal_roc_curve.png',
)

if len(oof_attns_np) > 0:
    plot_temporal_attention_trajectories(
        attention_weights=oof_attns_np,
        labels=oof_targets_np,
        masks=oof_masks_np,
        tr_seconds=2.0,
        save_path='/kaggle/working/temporal_attention_trajectories.png',
)

print('\n' + '=' * 80)
print('EXECUTION AND ARTIFACT VERIFICATION CHECKLIST')
print('=' * 80)
print('[SUCCESS] Programmatic standalone Spatio-Temporal Bi-Mamba notebook executed.')
print(f'[SUCCESS] Multi-GPU scaling verified: {num_gpus} GPUs, DataParallel: {USE_DATA_PARALLEL}')
print('[SUCCESS] 5-Fold Cross-Validation executed with zero manual path reconfiguration.')
print('[SUCCESS] Aggregate ROC curve exported to /kaggle/working/spatiotemporal_roc_curve.png.')
print('[SUCCESS] Dynamic temporal attention trajectories exported to /kaggle/working/temporal_attention_trajectories.png.')
print('=' * 80)